# Scaling with Number of Requested Eigenvalues

**Sweep:** `n_eigs`  
**Question:** How large must the basis be to accurately compute k eigenvalues? Does the required basis size scale linearly with k?

**Sweep variables:**
- `n_eigs` ∈ {5, 10, 15, 20, 25}
- `n_basis` ∈ {50, 100, 200, 400}

**Fixed:** `fb_fraction = 0.5`, `rtol = 1e-12`

**Domains:** rect, L_shape, GWW1 (all have reference eigenvalues)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import pandas as pd
import nb_utils

nb_utils.set_publication_style()
RESULTS_DIR = os.path.abspath('../results')

df, raw = nb_utils.load_sweep('n_eigs', RESULTS_DIR)
print(f'Loaded {len(df)} results')
print('Domains:', df['domain_name'].unique())
print('n_eigs values:', sorted(df['n_eigs'].unique()))
print('n_basis values:', sorted(df['n_basis'].unique()))
df.head(3)

## Plot 1: Error vs Basis Size for Fixed n_eigs

Each panel shows one domain. Lines are coloured by the number of requested eigenvalues k.

In [ ]:
domains = sorted(df['domain_name'].unique())
colors = nb_utils.domain_color_map(domains)
n_eigs_vals = sorted(df['n_eigs'].unique())
n_basis_vals = sorted(df['n_basis'].unique())

k_cmap = cm.get_cmap('plasma', len(n_eigs_vals))
k_colors = {k: k_cmap(i / (len(n_eigs_vals) - 1)) for i, k in enumerate(n_eigs_vals)}

fig, axes = plt.subplots(1, len(domains), figsize=(5 * len(domains), 4), sharey=False)
if len(domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, domains):
    for k in n_eigs_vals:
        d = df[(df['domain_name'] == dom) & (df['n_eigs'] == k)].sort_values('n_basis')
        if d['max_rel_error'].notna().any():
            ax.semilogy(d['n_basis'], d['max_rel_error'],
                        marker='o', markersize=5,
                        color=k_colors[k], label=f'k={k}')
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('$n_{\\mathrm{basis}}$', fontsize=9)
    ax.set_ylabel('Max relative error', fontsize=9)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(n_eigs_vals),
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig.suptitle('Convergence vs basis size for each n_eigs requested', fontsize=11)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

## Plot 2: Required Basis Size for Accuracy Threshold

For a target accuracy of `max_rel_error < 1e-8`, what is the minimum basis size needed as a function of k? If the relationship is linear, the required basis size scales proportionally with the number of requested eigenvalues.

In [ ]:
THRESHOLD = 1e-8

fig, ax = plt.subplots(figsize=(7, 4))
for dom in domains:
    min_nb_per_k = []
    for k in n_eigs_vals:
        d = df[(df['domain_name'] == dom) & (df['n_eigs'] == k)].sort_values('n_basis')
        achieved = d[d['max_rel_error'] < THRESHOLD]['n_basis']
        min_nb_per_k.append(achieved.min() if len(achieved) > 0 else np.nan)
    ax.plot(n_eigs_vals, min_nb_per_k,
            marker='o', color=colors[dom],
            label=nb_utils.label_domain(dom))

# Reference linear scaling line
k_ref = np.array(n_eigs_vals)
ax.plot(k_ref, 10 * k_ref, 'k--', linewidth=0.8, alpha=0.5, label='Linear (slope 10)')

ax.set_xlabel('Eigenvalues requested (k)')
ax.set_ylabel(f'Min n_basis for max_rel_error < {THRESHOLD:.0e}')
ax.set_title('Required basis size vs number of eigenvalues')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Plot 3: Accuracy Heatmap (n_eigs × n_basis)

Each cell shows `max_rel_error` for a given (n_eigs, n_basis) pair. Green = accurate, red = inaccurate.

In [ ]:
fig, axes = plt.subplots(1, len(domains), figsize=(5.5 * len(domains), 4))
if len(domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, domains):
    sub = df[df['domain_name'] == dom]
    pivot = sub.pivot_table(index='n_basis', columns='n_eigs',
                            values='max_rel_error', aggfunc='min')
    log_pivot = np.log10(pivot.values + 1e-20)

    im = ax.imshow(log_pivot, aspect='auto', cmap='RdYlGn_r',
                   vmin=-14, vmax=-2, origin='lower')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(r) for r in pivot.index], fontsize=8)
    ax.set_xlabel('n_eigs (k)', fontsize=9)
    ax.set_ylabel('n_basis', fontsize=9)
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)

    # Annotate cells
    for i in range(log_pivot.shape[0]):
        for j in range(log_pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.0e}', ha='center', va='center', fontsize=7)

    fig.colorbar(im, ax=ax, label='log₁₀(max rel. error)', fraction=0.04)

fig.suptitle('Accuracy heatmap: n_basis × n_eigs', fontsize=11)
plt.tight_layout()
plt.show()

## Plot 4: Wall Time vs Basis Size

Is wall time dominated by basis size or by the number of eigenvalues requested?

In [ ]:
domain_markers = {'rect': 'o', 'L_shape': 's', 'GWW1': '^'}

fig, ax = plt.subplots(figsize=(7, 4.5))
for k in n_eigs_vals:
    for dom in domains:
        d = df[(df['domain_name'] == dom) & (df['n_eigs'] == k)].sort_values('n_basis')
        ax.scatter(d['n_basis'], d['wall_time'],
                   c=[k_colors[k]] * len(d),
                   marker=domain_markers.get(dom, 'o'),
                   s=50, alpha=0.8)

# Legend: colours = n_eigs, markers = domain
from matplotlib.lines import Line2D
k_handles = [Line2D([0], [0], color=k_colors[k], marker='o', linestyle='',
                    label=f'k={k}') for k in n_eigs_vals]
dom_handles = [Line2D([0], [0], color='gray',
                      marker=domain_markers.get(d, 'o'), linestyle='',
                      label=nb_utils.label_domain(d)) for d in domains]
ax.legend(handles=k_handles + dom_handles, fontsize=8, ncol=2)

ax.set_xlabel('$n_{\\mathrm{basis}}$')
ax.set_ylabel('Wall time (s)')
ax.set_title('Computation cost vs basis size (colour = k, marker = domain)')
plt.tight_layout()
plt.show()

## Plot 5: Per-Eigenvalue Errors at n_basis = 200

Do higher-index eigenvalues have consistently larger errors, regardless of how many were requested?

In [ ]:
TARGET_N = 200

# index: (domain, n_eigs, n_basis) → raw result
index = {}
for r in raw:
    cfg = r.config
    nb = cfg.n_fb + cfg.n_fs
    index[(cfg.domain_name, cfg.n_eigs, nb)] = r

fig, axes = plt.subplots(1, len(domains), figsize=(5 * len(domains), 4))
if len(domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, domains):
    for k in n_eigs_vals:
        r = index.get((dom, k, TARGET_N))
        if r is None or r.rel_errors is None:
            continue
        eig_idx = np.arange(1, len(r.rel_errors) + 1)
        ax.semilogy(eig_idx, r.rel_errors,
                    marker='o', markersize=4,
                    color=k_colors[k], label=f'k={k}')
    ax.set_title(nb_utils.label_domain(dom), fontsize=9)
    ax.set_xlabel('Eigenvalue index (absolute)', fontsize=9)
    ax.set_ylabel('Relative error', fontsize=9)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(n_eigs_vals),
           fontsize=9, bbox_to_anchor=(0.5, -0.08))
fig.suptitle(f'Per-eigenvalue errors at n_basis = {TARGET_N}', fontsize=11)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()